# Predict the pH of a Simple Aqueous Solution — KH₂PO₄ + NH₄Cl

**The situation:** you're making up a defined liquid growth medium from
two of its most common ingredients — KH₂PO₄ (a phosphate buffer salt) and
NH₄Cl (a nitrogen source) — dissolved in water. There is no gas headspace
to model and no precipitating solid to track, just the liquid phase on its
own. What pH does that land at, and how sensitive is it to how much of
each salt you weigh out?

This is the smallest possible use of PyOMES's chemical-equilibrium layer:
one engine, one reaction network, one `solve()` call. No `Phase`,
`ControlVolume`, or `Simulation` objects are needed for a single
equilibrium snapshot like this — those come in once the chemistry needs to
evolve over time or couple to other phases (see
[`demos/model_api/chemistry/reaction_system.py`](../model_api/chemistry/reaction_system.py)
for that step).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

sys.path.insert(0, str(_find_repo() / "models"))

from PyOMES.chemistry.common_species import (
    H2O, H_plus, OH_minus,
    H3PO4, H2PO4_minus, HPO4_2minus, PO4_3minus,
    NH3, NH4_plus,
)
from PyOMES.reactions.equilibrium import EquilibriumReaction
from PyOMES.reactions.stoichiometry import StoichiometryEntry
from PyOMES.chemical_equilibrium.nr_engine import NRChemicalEquilibriumEngine

def _e(sp, coeff):
    return StoichiometryEntry(species=sp, phase="liquid", coefficient=coeff)

print("Imports OK")


## 1  Declare the chemistry

Both salts fully dissociate on dissolving — KH₂PO₄ → K⁺ + H₂PO₄⁻, and
NH₄Cl → NH₄⁺ + Cl⁻ — so the acid-base chemistry actually at play is the
phosphate ladder (from the H₂PO₄⁻ that KH₂PO₄ contributes) and the
ammonium/ammonia couple (from the NH₄⁺ that NH₄Cl contributes), on top of
water's own autoionization. K⁺ and Cl⁻ take no part in any reaction — they
only enter the charge balance, as `strong_ions=`.

| Reaction | log K | p$K_a$ | Role |
|---|---|---|---|
| H₂O ⇌ H⁺ + OH⁻ | −14.0 | 14.0 | water autoionization |
| H₃PO₄ ⇌ H₂PO₄⁻ + H⁺ | −2.15 | 2.15 | phosphate, 1st step |
| H₂PO₄⁻ ⇌ HPO₄²⁻ + H⁺ | −7.20 | 7.20 | phosphate, 2nd step |
| HPO₄²⁻ ⇌ PO₄³⁻ + H⁺ | −12.35 | 12.35 | phosphate, 3rd step |
| NH₄⁺ ⇌ NH₃ + H⁺ | −9.25 | 9.25 | ammonium/ammonia |

`total_id="H3PO4"` on the three phosphate steps and `total_id="NH3"` on
the ammonium step tell the engine which mass-balance total each species
belongs to — the total phosphate and total ammoniacal nitrogen you'll
supply to `solve()` later. Note that the *master* label (`H3PO4`, `NH3`)
is just an id for the total; it doesn't imply the solution actually starts
from those molecular forms — here it starts from H₂PO₄⁻ and NH₄⁺, which
the engine handles identically since it solves for equilibrium composition,
not a synthesis path.

In [ ]:
water = EquilibriumReaction(
    stoichiometry=[_e(H2O, -1), _e(H_plus, +1), _e(OH_minus, +1)],
    log_K=-14.0, label="water",
)
p1 = EquilibriumReaction(
    stoichiometry=[_e(H3PO4, -1), _e(H2PO4_minus, +1), _e(H_plus, +1)],
    log_K=-2.15, total_id="H3PO4", label="p1",
)
p2 = EquilibriumReaction(
    stoichiometry=[_e(H2PO4_minus, -1), _e(HPO4_2minus, +1), _e(H_plus, +1)],
    log_K=-7.20, total_id="H3PO4", label="p2",
)
p3 = EquilibriumReaction(
    stoichiometry=[_e(HPO4_2minus, -1), _e(PO4_3minus, +1), _e(H_plus, +1)],
    log_K=-12.35, total_id="H3PO4", label="p3",
)
nh4 = EquilibriumReaction(
    stoichiometry=[_e(NH4_plus, -1), _e(NH3, +1), _e(H_plus, +1)],
    log_K=-9.25, total_id="NH3", label="nh4",
)

print("5 reactions declared.")


## 2  Instantiate the equilibrium engine

`NRChemicalEquilibriumEngine.from_reactions()` takes the flat list of
reactions and builds a Newton-Raphson tableau from them — this is the
"chemicalEquilibrium system" for this sample: a solver object that knows
the reaction network and can be asked to equilibrate at any composition.

In [ ]:
engine = NRChemicalEquilibriumEngine.from_reactions([water, p1, p2, p3, nh4])

print("Masters:    ", engine.tableau.masters)
print("Secondaries:", [s.species_id for s in engine.tableau.secondaries])


## 3  Solve for the sample's pH

A concrete recipe close to M9 minimal medium: 22 mmol/L KH₂PO₄ and
18.7 mmol/L NH₄Cl. Total phosphate and total ammoniacal nitrogen go in via
`totals=`, keyed by master id; the K⁺ and Cl⁻ each salt contributes go in
via `strong_ions=` at the *same* concentration as the parent salt, since
both dissociate 1:1.

In [ ]:
CT_P = 0.022   # mol/L KH2PO4 -> mol/L total phosphate, mol/L K+
CT_N = 0.0187  # mol/L NH4Cl  -> mol/L total ammoniacal N, mol/L Cl-

result = engine.solve(
    totals={"H3PO4": CT_P, "NH3": CT_N},
    strong_ions={"CT_K": CT_P, "CT_Cl": CT_N},
)

print(f"pH               = {result.pH:.3f}")
print(f"[H2PO4-]         = {result.species_mol_L['H2PO4-']*1e3:.4f} mmol/L")
print(f"[HPO4--]         = {result.species_mol_L['HPO4--']*1e6:.4f} µmol/L")
print(f"[NH4+]           = {result.species_mol_L['NH4+']*1e3:.4f} mmol/L")
print(f"[NH3]            = {result.species_mol_L['NH3']*1e6:.4f} µmol/L")
print(f"charge residual  = {result.charge_residual:.2e}  (should be ~0)")


## 4  Sensitivity to each salt in isolation

Before the 2-D picture, it's worth seeing each salt's effect on its own.
KH₂PO₄ alone sets an *amphoteric* pH — H₂PO₄⁻ is both a weak acid (toward
HPO₄²⁻) and a weak base (toward H₃PO₄), and at moderate-to-high
concentration the solution pH converges toward
$\tfrac12(\text{p}K_{a1}+\text{p}K_{a2}) \approx 4.68$, largely
independent of exactly how much is dosed. NH₄Cl on its own is a much
weaker acid (p$K_a$ = 9.25, far from neutral) so its effect on pH near
neutral-to-acidic conditions is comparatively small.

In [ ]:
CT_P_vals = np.logspace(-3, -1, 30)    # 1 - 100 mmol/L KH2PO4
pH_P = [engine.solve(totals={"H3PO4": CT, "NH3": 0.0},
                     strong_ions={"CT_K": CT}).pH
        for CT in CT_P_vals]

CT_N_vals = np.logspace(-3, -1, 30)    # 1 - 100 mmol/L NH4Cl
pH_N = [engine.solve(totals={"H3PO4": 0.0, "NH3": CT},
                     strong_ions={"CT_Cl": CT}).pH
        for CT in CT_N_vals]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

axes[0].semilogx(CT_P_vals * 1e3, pH_P, "o-", color="tab:blue")
axes[0].axhline(0.5 * (2.15 + 7.20), color="gray", ls="--", lw=1,
                label=r"$\frac{1}{2}(pK_{a1}+pK_{a2})=4.68$")
axes[0].set_xlabel("KH2PO4 (mmol/L)")
axes[0].set_ylabel("pH")
axes[0].set_title("KH2PO4 alone")
axes[0].legend(fontsize=8)
axes[0].grid(True, which="both", alpha=0.3)

axes[1].semilogx(CT_N_vals * 1e3, pH_N, "o-", color="tab:orange")
axes[1].set_xlabel("NH4Cl (mmol/L)")
axes[1].set_title("NH4Cl alone")
axes[1].grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()


## 5  pH across the KH₂PO₄ / NH₄Cl growth-media design space

Putting both salts together, over the range of doses commonly used in
defined bacterial/yeast growth media (roughly 1-100 mmol/L for each —
M9 minimal medium, for instance, uses ~22 mmol/L KH₂PO₄ and
~19 mmol/L NH₄Cl), gives a 2-D picture of how the recipe determines pH.
The result confirms what Section 4 suggested: KH₂PO₄ concentration is the
dominant control — contour lines run mostly horizontal — while NH₄Cl only
measurably shifts pH when phosphate is comparatively dilute.

The plot itself is deferred to the end of §6: it's drawn together with the
PHREEQC parity comparison as a single two-panel publication figure once
that comparison data exists, so the two share one figure style instead of
each getting its own throwaway version.

In [ ]:
n_grid = 40
CT_P_grid = np.logspace(-3, -1, n_grid)   # 1 - 100 mmol/L KH2PO4
CT_N_grid = np.logspace(-3, -1, n_grid)   # 1 - 100 mmol/L NH4Cl

pH_grid = np.empty((n_grid, n_grid))
for i, ct_n in enumerate(CT_N_grid):
    for j, ct_p in enumerate(CT_P_grid):
        out = engine.solve(
            totals={"H3PO4": ct_p, "NH3": ct_n},
            strong_ions={"CT_K": ct_p, "CT_Cl": ct_n},
        )
        pH_grid[i, j] = out.pH

print(f"pH range over the grid: {pH_grid.min():.3f} - {pH_grid.max():.3f}")


## 6  Benchmark against PHREEQC

Everything above solves a hand-declared reaction network at infinite
dilution — no activity correction. [PHREEQC](https://www.usgs.gov/software/phreeqc-version-3)
is the de facto standard geochemical speciation code; comparing against
it checks both the NR solver itself and where the infinite-dilution
assumption used in Sections 1-5 starts to matter.

**Which PHREEQC formulation, specifically:** `PHREEQCChemicalEquilibriumEngine`
goes through `phreeqpython`, whose default database is `vitens.dat` — a
derivative of the standard `phreeqc.dat` (same reactions, same activity
treatment). That database is **not** ideal: every species carries a
`-gamma` (ion-size å, b-dot) entry, so PHREEQC applies the **extended
(WATEQ) Debye-Hückel equation** to every solve, and its reaction list
includes the `KHPO4⁻`/`NaHPO4⁻` ion pairs. So this section compares two
*different nonideal* activity treatments against each other (WATEQ
Debye-Hückel + ion pairing vs. PyOMES's Davies option), not a nonideal
model against an ideal one — the notebook's own `engine` (ideal, Sections
1-5) is plotted alongside purely as the "what if we correct for nothing"
baseline.

This section is optional — it needs the `phreeqpython` package
(`pip install PyOMES[phreeqc]`). If it isn't installed, the cell below
reports that and the rest of the notebook is unaffected.

In [ ]:
try:
    from PyOMES.chemical_equilibrium.phreeqc_engine import PHREEQCChemicalEquilibriumEngine
    _HAVE_PHREEQC = True
    print("phreeqpython available - PHREEQC benchmark cells will run.")
except ImportError as exc:
    _HAVE_PHREEQC = False
    print(f"phreeqpython not installed ({exc}); skipping PHREEQC benchmark cells.")
    print("Install with: pip install PyOMES[phreeqc]")


### 6a  A single point: the M9-like recipe

Three engines, same recipe as Section 3:

- **`engine` (ideal)** — the engine used throughout this notebook; no
  activity correction, strictly valid only at infinite dilution.
- **`engine_davies`** — identical reaction network, with
  `use_activity=True, activity_model="davies"`.
- **PHREEQC**, via `PHREEQCChemicalEquilibriumEngine` — `component_map`
  routes `H3PO4`→`P`, `NH3`→`N(-3)`, `CT_K`→`K`, `CT_Cl`→`Cl`. As noted
  above, this is PHREEQC running its own **nonideal** default (extended
  WATEQ Debye-Hückel activities, plus the `KHPO4⁻` ion pair PyOMES doesn't
  model) — not an ideal reference.

`use_warmstart=False` is required here — the default `True` reuses the
PHREEQC solution across calls *incrementally* (a `REACTION` addition, not
a reset to the declared total), which would double-count the composition
already used to prime the engine. See
[`demos/features/ChemicalEquilibriumProtocol/03_phreeqc_engine_basics.ipynb`](../features/ChemicalEquilibriumProtocol/03_phreeqc_engine_basics.ipynb)
§4 for the mechanism.

In [ ]:
if _HAVE_PHREEQC:
    engine_davies = NRChemicalEquilibriumEngine.from_reactions(
        [water, p1, p2, p3, nh4], use_activity=True, activity_model="davies",
    )
    engine_pq = PHREEQCChemicalEquilibriumEngine(
        {"H3PO4": CT_P * 1e3, "NH3": CT_N * 1e3, "CT_K": CT_P * 1e3, "CT_Cl": CT_N * 1e3},
        component_map={"H3PO4": "P", "NH3": "N(-3)", "CT_K": "K", "CT_Cl": "Cl"},
        use_warmstart=False,
    )

    result_davies = engine_davies.solve(
        totals={"H3PO4": CT_P, "NH3": CT_N}, strong_ions={"CT_K": CT_P, "CT_Cl": CT_N},
    )
    result_pq = engine_pq.solve(
        totals={"H3PO4": CT_P, "NH3": CT_N, "CT_K": CT_P, "CT_Cl": CT_N},
    )

    print(f"{'Engine':<28}{'pH':>8}")
    print(f"{'NR (ideal)':<28}{result.pH:>8.3f}")
    print(f"{'NR (Davies activity)':<28}{result_davies.pH:>8.3f}")
    print(f"{'PHREEQC (WATEQ Debye-Huckel)':<28}{result_pq.pH:>8.3f}")
    print()
    print(f"|ideal  - PHREEQC| = {abs(result.pH - result_pq.pH):.3f} pH units")
    print(f"|Davies - PHREEQC| = {abs(result_davies.pH - result_pq.pH):.3f} pH units")


### 6b  Where the infinite-dilution assumption breaks down

Repeating Section 4's KH₂PO₄-alone sweep with all three engines. Ionic
strength grows with dose (K⁺ and H₂PO₄⁻ both scale with `CT_P`), so the
ideal engine's error against PHREEQC (WATEQ Debye-Hückel) should grow
with concentration too, while the Davies-corrected engine — a different
nonideal treatment, not the same equation PHREEQC uses — should still
track it closely across the whole range. The left panel shows pH vs.
dose directly; the right panel is a parity plot (PyOMES pH vs. PHREEQC
pH at matching doses — perfect agreement falls on the dashed 1:1 line).

In [ ]:
if _HAVE_PHREEQC:
    pH_P_davies = [engine_davies.solve(totals={"H3PO4": CT, "NH3": 0.0},
                                        strong_ions={"CT_K": CT}).pH
                   for CT in CT_P_vals]

    engine_pq_sweep = PHREEQCChemicalEquilibriumEngine(
        {"H3PO4": 1.0, "CT_K": 1.0}, component_map={"H3PO4": "P", "CT_K": "K"},
        use_warmstart=False,
    )
    pH_P_pq = np.array([engine_pq_sweep.solve(totals={"H3PO4": CT, "CT_K": CT}).pH
                         for CT in CT_P_vals])
    pH_P_arr = np.array(pH_P)
    pH_P_davies_arr = np.array(pH_P_davies)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    axes[0].semilogx(CT_P_vals * 1e3, pH_P_arr, "o-", color="tab:blue", label="NR (ideal)")
    axes[0].semilogx(CT_P_vals * 1e3, pH_P_davies_arr, "^-", color="tab:green", label="NR (Davies)")
    axes[0].semilogx(CT_P_vals * 1e3, pH_P_pq, "s--", color="k", label="PHREEQC (WATEQ D-H)")
    axes[0].set_xlabel("KH2PO4 (mmol/L)")
    axes[0].set_ylabel("pH")
    axes[0].set_title("KH2PO4 alone - three engines")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, which="both", alpha=0.3)

    lo = min(pH_P_pq.min(), pH_P_arr.min(), pH_P_davies_arr.min())
    hi = max(pH_P_pq.max(), pH_P_arr.max(), pH_P_davies_arr.max())
    pad = (hi - lo) * 0.08
    axes[1].plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=1, zorder=0, label="1:1")
    axes[1].scatter(pH_P_pq, pH_P_arr, s=28, color="tab:blue", alpha=0.8, label="NR (ideal)")
    axes[1].scatter(pH_P_pq, pH_P_davies_arr, s=28, color="tab:green", marker="^", alpha=0.8, label="NR (Davies)")
    axes[1].set_xlabel("PHREEQC pH (WATEQ D-H)")
    axes[1].set_ylabel("PyOMES pH")
    axes[1].set_title("Parity vs. PHREEQC")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"Max |ideal  - PHREEQC| = {np.max(np.abs(pH_P_arr - pH_P_pq)):.3f} pH units")
    print(f"Max |Davies - PHREEQC| = {np.max(np.abs(pH_P_davies_arr - pH_P_pq)):.3f} pH units")


The ideal engine's disagreement with PHREEQC grows with concentration, as
expected — activity effects strengthen with ionic strength, and the ideal
engine applies none. The Davies-corrected engine sits close to the 1:1
line across the whole range even though it isn't using PHREEQC's own
activity equation; the small residual that remains is a real model
difference (PHREEQC's `KHPO4⁻` ion pair, which this notebook's reaction
network doesn't declare, plus Davies vs. WATEQ Debye-Hückel being
different empirical fits), not solver error.

But that raises a fair question: how much of the 6a/6b gap is *really*
activity theory, versus PHREEQC simply using different log K values or
extra reactions? Section 6c isolates that.

### 6c  Isolating the effect: forcing PHREEQC to (near-)ideal too

PHREEQC has **no built-in switch** for "ideal solution" — every stock
database applies an activity correction unconditionally. The standard
workaround (a documented PHREEQC technique, not a PyOMES feature) is to
override each species' `-gamma` ion-size parameter with something huge
and its `b`-dot term with 0: in the extended Debye-Hückel equation
`log10(γ) = -A z² √I / (1 + B a √I) + b I`, driving `a → ∞` drives the
whole correction to 0, i.e. `γ → 1`.

Doing this cleanly means writing a **from-scratch minimal database**
(rather than patching `vitens.dat`'s existing 900+ species, which are
wired together through a master-species graph that doesn't tolerate
piecemeal overrides) with just the species this notebook needs. That has
a second benefit: the database's log K values can be set to *exactly*
VLsim's own (2.15, 7.20, 12.35, 9.25, 14.0), removing thermodynamic-data
differences as a confound entirely. What's left, if anything, is pure
numerical-solver disagreement — the cleanest possible check of the NR
solver itself.

In [ ]:
if _HAVE_PHREEQC:
    import tempfile
    from phreeqpython import PhreeqPython as _RawPhreeqPython

    # Same 5 reactions and log K's as Section 1, written as a PHREEQC database.
    # -gamma <huge> 0  suppresses the extended Debye-Hueckel term (gamma -> 1)
    # on every charged species -- an approximation of "ideal", not a native
    # PHREEQC mode.  N is declared as a plain (non-redox) master species since
    # this system never touches another nitrogen oxidation state.
    _IDEAL_DB = """\
SOLUTION_MASTER_SPECIES
H       H+      -1.     H       1.008
H(0)    H2      0.0     H
H(1)    H+      -1.     0.0
E       e-      0.0     0.0     0.0
O       H2O     0.0     O       16.00
O(0)    O2      0.0     O
O(-2)   H2O     0.0     0.0
P       PO4-3   0.0     P       30.974
N       NH4+    0.0     N       14.0067
K       K+      0.0     K       39.098
Cl      Cl-     0.0     Cl      35.453

SOLUTION_SPECIES
H+ = H+
        log_k           0.0
        -gamma          1e6     0
e- = e-
        log_k           0.0
H2O = H2O
        log_k           0.0
2 H+ + 2 e- = H2
        log_k           -3.15
2 H2O = O2 + 4 H+ + 4 e-
        log_k           -86.08
H2O = OH- + H+
        log_k           -14.0
        -gamma          1e6     0
PO4-3 = PO4-3
        log_k           0.0
        -gamma          1e6     0
PO4-3 + H+ = HPO4-2
        log_k           12.35
        -gamma          1e6     0
PO4-3 + 2H+ = H2PO4-
        log_k           19.55
        -gamma          1e6     0
PO4-3 + 3H+ = H3PO4
        log_k           21.70
NH4+ = NH4+
        log_k           0.0
        -gamma          1e6     0
NH4+ = NH3 + H+
        log_k           -9.25
K+ = K+
        log_k           0.0
        -gamma          1e6     0
Cl- = Cl-
        log_k           0.0
        -gamma          1e6     0
END
"""
    _db_dir = Path(tempfile.gettempdir())
    (_db_dir / "vlsim_ideal_phosphate_ammonium.dat").write_text(_IDEAL_DB)
    pp_ideal = _RawPhreeqPython(database="vlsim_ideal_phosphate_ammonium.dat",
                                 database_directory=_db_dir)

    def _solve_ideal_pq(ct_p, ct_n):
        sol = pp_ideal.add_solution_raw({
            "P": ct_p * 1e3, "N": ct_n * 1e3, "K": ct_p * 1e3, "Cl": ct_n * 1e3,
            "temp": 25.0, "pH": "7 charge", "units": "mmol/L",
        })
        ph = sol.pH
        sol.forget()
        return ph

    ph_point_ideal_pq = _solve_ideal_pq(CT_P, CT_N)
    print(f"NR (ideal)              pH = {result.pH:.4f}")
    print(f"PHREEQC (gamma -> 1)    pH = {ph_point_ideal_pq:.4f}")
    print(f"|NR ideal - PHREEQC ideal| = {abs(result.pH - ph_point_ideal_pq):.4f} pH units")


Repeating the KH₂PO₄-alone sweep once more, ideal engine against
near-ideal PHREEQC with matched log K's — this is the true solver-vs-solver
check the earlier sections couldn't isolate.

In [ ]:
if _HAVE_PHREEQC:
    pH_P_ideal_pq = np.array([_solve_ideal_pq(ct, 0.0) for ct in CT_P_vals])

    fig, ax = plt.subplots(figsize=(5.5, 5))
    lo = min(pH_P_arr.min(), pH_P_ideal_pq.min())
    hi = max(pH_P_arr.max(), pH_P_ideal_pq.max())
    pad = (hi - lo) * 0.08
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=1, zorder=0, label="1:1")
    ax.scatter(pH_P_ideal_pq, pH_P_arr, s=28, color="tab:red", alpha=0.8)
    ax.set_xlabel("PHREEQC pH (gamma -> 1, matched log K)")
    ax.set_ylabel("NR (ideal) pH")
    ax.set_title("Ideal vs. (near-)ideal: pure solver agreement")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Max |NR ideal - PHREEQC ideal| = {np.max(np.abs(pH_P_arr - pH_P_ideal_pq)):.4f} pH units")
    print(f"(compare: Max |NR ideal - PHREEQC WATEQ D-H| = {np.max(np.abs(pH_P_arr - pH_P_pq)):.3f} pH units, Section 6b)")


With activity corrections and reaction data both pinned to "ideal, same
log K's," the two solvers agree to within a few thousandths of a pH
unit across the whole dose range — roughly 50x tighter than the
ideal-vs-real-PHREEQC gap from Section 6b. That confirms the 6a/6b
mismatch is (almost entirely) activity theory, not a bug in either
solver.

### 6d  Publication figures: design space + accuracy, as separate files

Same shared figure style as before, but each panel is now its own
independent figure, saved as its own high-resolution file (vector PDF +
600dpi PNG) so each can be dropped into a LaTeX document directly (e.g.
`\includegraphics{fig_contour_ph_design_space.pdf}`):

- `fig_contour_ph_design_space.{pdf,png}` — the KH₂PO₄/NH₄Cl design-space
  contour from Section 5 (deferred to here so it could be built and styled
  together with the parity figure below).
- `fig_parity_vs_phreeqc.{pdf,png}` — the PHREEQC parity comparison, only
  produced if `phreeqpython` is installed — PyOMES vs. PHREEQC under
  *matching* activity treatments in each case: ideal vs. ideal (Section 6c)
  and Davies vs. WATEQ Debye-Hückel (Section 6b). Each series lands close
  to the 1:1 line on its own terms; the point is that "close" means
  something very different for the two — thousandths of a pH unit for the
  ideal pair, hundredths for the nonideal pair (the residual there being
  real model differences: Davies vs. WATEQ Debye-Hückel are different
  empirical fits, and PHREEQC's `KHPO4⁻` ion pair isn't in this notebook's
  reaction network).

In [ ]:
from matplotlib.ticker import MultipleLocator

# One shared style for both figures, each sized for a single-column
# placement (~3.5 in / 89 mm wide) in a two-column article. Scoped with
# rc_context so it doesn't leak into any other plot in this notebook.
FS_ticks = 10
FS_axes = 12

PUB_FIG_WIDTH_IN = 3.5
PUB_DPI = 600
PUB_STYLE = {
    "font.size": 8,
    "axes.labelsize": FS_axes,
    "axes.titlesize": FS_axes,
    "xtick.labelsize": FS_ticks,
    "ytick.labelsize": FS_ticks,
    "legend.fontsize": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "grid.linewidth": 0.5,
}

FIG_DIR = _find_repo() / "demos" / "usecases" / "figures"
FIG_DIR.mkdir(exist_ok=True)

with plt.rc_context(PUB_STYLE):
    # -- figure 1: design-space contour, own file ------------------------
    # A slim extra row (not a side column) holds a horizontal colorbar,
    # placed below the contour panel, so the panel can use the full
    # figure width.
    fig_a = plt.figure(figsize=(PUB_FIG_WIDTH_IN, 3.9), layout="constrained")
    gs_a = fig_a.add_gridspec(2, 1, height_ratios=[10, 0.6])
    ax_a = fig_a.add_subplot(gs_a[0, 0])
    cax_a = fig_a.add_subplot(gs_a[1, 0])
    fig_a.get_layout_engine().set(hspace=0.08)

    cf = ax_a.contourf(CT_P_grid * 1e3, CT_N_grid * 1e3, pH_grid,
                        levels=20, cmap="viridis")
    # Standard 0.1-pH-unit contour lines (5.1, 5.0, 4.9, ...) instead of
    # data-range-derived levels.
    lo, hi = pH_grid.min(), pH_grid.max()
    line_levels = np.round(np.arange(np.ceil(lo * 10) / 10, hi, 0.1), 1)
    cs = ax_a.contour(CT_P_grid * 1e3, CT_N_grid * 1e3, pH_grid,
                       levels=line_levels, colors="white", linewidths=0.5)
    ax_a.clabel(cs, inline=True, fontsize=9, fmt="%.1f")
    ax_a.set_xscale("log")
    ax_a.set_yscale("log")
    ax_a.set_xlabel("KH2PO4 (mmol/L)")
    ax_a.set_ylabel("NH4Cl (mmol/L)")
    cbar = fig_a.colorbar(cf, cax=cax_a, orientation="horizontal")
    # Same standardized 0.1-pH ticks as the contour line labels, rather
    # than data-range-derived (and less round-looking) values.
    cbar.set_ticks(line_levels)
    cbar.set_label("pH")

    for ext in ("pdf", "png"):
        fig_a.savefig(FIG_DIR / f"fig_contour_ph_design_space.{ext}",
                      dpi=PUB_DPI, bbox_inches="tight")
    plt.show()

    # -- figure 2: PHREEQC parity, own file, if available ----------------
    if _HAVE_PHREEQC:
        fig_b, ax_b = plt.subplots(figsize=(PUB_FIG_WIDTH_IN, 3.3), layout="constrained")

        # Marker colors sampled from the same viridis colormap as the
        # contour figure above, so the two share one palette.
        color_ideal, color_nonideal = plt.cm.viridis([0.15, 0.85])

        all_vals = np.concatenate([pH_P_ideal_pq, pH_P_arr, pH_P_pq, pH_P_davies_arr])
        lo, hi = all_vals.min(), all_vals.max()
        pad = (hi - lo) * 0.05
        ax_b.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=0.8,
                  zorder=0, label="1:1")
        ax_b.scatter(pH_P_ideal_pq, pH_P_arr, s=16, color=color_ideal, alpha=0.8,
                     label="Ideal")
        ax_b.scatter(pH_P_pq, pH_P_davies_arr, s=16, color=color_nonideal, marker="^",
                     alpha=0.8, label="Nonideal")
        ax_b.set_xlabel("PHREEQC pH")
        ax_b.set_ylabel("PyOMES pH")
        # Same range on both axes (not just matplotlib's independent
        # per-axis autoscale) so the 1:1 line is a true 45 degrees.
        ax_b.set_xlim(lo - pad, hi + pad)
        ax_b.set_ylim(lo - pad, hi + pad)
        ax_b.set_aspect("equal", adjustable="box")
        # Equal xlim/ylim alone doesn't guarantee equal ticks: the default
        # locator picks tick spacing per axis based on the box's pixel
        # size, which "equal" aspect can make unequal (x got 0.2 spacing,
        # y got 0.1). Force both to the same explicit spacing.
        ax_b.xaxis.set_major_locator(MultipleLocator(0.1))
        ax_b.yaxis.set_major_locator(MultipleLocator(0.1))
        ax_b.legend(loc="lower right", frameon=True, handletextpad=0.4,
                    borderpad=0.4, labelspacing=0.3)

        for ext in ("pdf", "png"):
            fig_b.savefig(FIG_DIR / f"fig_parity_vs_phreeqc.{ext}",
                          dpi=PUB_DPI, bbox_inches="tight")
        plt.show()

if _HAVE_PHREEQC:
    print(f"Max |ideal series|    (NR ideal  vs. PHREEQC gamma->1) = {np.max(np.abs(pH_P_arr - pH_P_ideal_pq)):.4f} pH units")
    print(f"Max |nonideal series| (NR Davies vs. PHREEQC WATEQ D-H) = {np.max(np.abs(pH_P_davies_arr - pH_P_pq)):.4f} pH units")
    print(f"Saved: {FIG_DIR / 'fig_contour_ph_design_space.pdf'}")
    print(f"Saved: {FIG_DIR / 'fig_parity_vs_phreeqc.pdf'}")
else:
    print(f"Saved: {FIG_DIR / 'fig_contour_ph_design_space.pdf'}")


For a much deeper accuracy audit — carbonate, calcium, precipitation —
see
[`demos/model_api/chemistry/speciation/06_phreeqc_benchmark.ipynb`](../model_api/chemistry/speciation/06_phreeqc_benchmark.ipynb).

## 7  Run time: software x activity model, 500 replicates each

One last comparison, this time on speed rather than accuracy — split by
*both* axes Section 6 already established, not just by software: the same
M9-like point from Section 6a (`CT_P`, `CT_N`), solved 500 times each by
all four engine/activity-model combinations from Sections 6a-6c (same
four-way split [`04_compare_runtime_by_usecase.ipynb`](04_compare_runtime_by_usecase.ipynb)
uses):

- **PyOMES ideal** (`engine`) — no activity correction.
- **PyOMES Davies** (`engine_davies`) — `use_activity=True, activity_model="davies"`.
- **PHREEQC default** (`engine_pq`) — `vitens.dat`'s WATEQ Debye-Hückel.
- **PHREEQC ideal-equivalent** (`_solve_ideal_pq`) — the §6c `-gamma 1e6 0`
  trick (γ→1) against the matched-log-K minimal database.

Each gets one untimed warmup call first so neither engine's one-time
import/IPC-connection cost biases the result, then mean ± standard
deviation over 500 timed replicate calls — not the noise-robust
"best-of-trials" timing usecase 04 uses, since the point here is to
characterize the *spread* of individual call times, not to filter it out.
The last three rows need `phreeqpython`; without it, only PyOMES ideal is
reported.

(PyOMES's engine was called `VLsim` in an earlier version of this project —
same engine, current name.)

In [ ]:
import time

N_REPS = 500

def time_replicates(fn, n=N_REPS):
    fn()  # untimed warmup -- first call pays one-time import/IPC costs
    times_s = np.empty(n)
    for i in range(n):
        t0 = time.perf_counter()
        fn()
        times_s[i] = time.perf_counter() - t0
    return times_s * 1e3  # ms

runtime_results = {}
runtime_results["PyOMES ideal"] = time_replicates(lambda: engine.solve(
    totals={"H3PO4": CT_P, "NH3": CT_N},
    strong_ions={"CT_K": CT_P, "CT_Cl": CT_N},
))

if _HAVE_PHREEQC:
    runtime_results["PyOMES Davies"] = time_replicates(lambda: engine_davies.solve(
        totals={"H3PO4": CT_P, "NH3": CT_N},
        strong_ions={"CT_K": CT_P, "CT_Cl": CT_N},
    ))
    runtime_results["PHREEQC default (WATEQ D-H)"] = time_replicates(lambda: engine_pq.solve(
        totals={"H3PO4": CT_P, "NH3": CT_N, "CT_K": CT_P, "CT_Cl": CT_N},
    ))
    runtime_results["PHREEQC ideal-equivalent (gamma->1)"] = time_replicates(
        lambda: _solve_ideal_pq(CT_P, CT_N)
    )
else:
    print("phreeqpython not installed; only PyOMES ideal timed below.")

print(f"{'Software / activity model':<36}{'mean (ms)':>12}{'std (ms)':>12}")
for label, times_ms in runtime_results.items():
    print(f"{label:<36}{times_ms.mean():>12.4f}{times_ms.std():>12.4f}")


## Where to go next

- **More chemistry in the same liquid** (strong ions as a matter of
  course, temperature correction, activity coefficients) —
  [`demos/model_api/chemistry/speciation/`](../model_api/chemistry/speciation/),
  which verifies this same engine against closed-form analytical results
  (including this exact phosphate ladder).
- **Engine mechanics and gotchas** (constructor arguments, warmstart
  caching, what `algebraic_species()` returns) —
  [`demos/features/ChemicalEquilibriumProtocol/`](../features/ChemicalEquilibriumProtocol/).
- **Wiring this into something that evolves over time** — a fermenter or
  reactor where pH is one state among many being integrated — see
  [`demos/model_api/chemistry/reaction_system.py`](../model_api/chemistry/reaction_system.py)
  and [`demos/builder/`](../builder/) for the full `Simulation` pattern.